# SUTV: City and Zipcode Prioritization

**Project:** Shape Up The Vote — 2024 Voter Engagement Initiative  
**Author:** Stann  
**Last Updated:** 2024-05-16

---

### Description

Shape Up The Vote will be launching its campaign in these states:

* **Wave 1 States:** AZ, GA, NV, MI, NC, PA, WI, OH  
* **Wave 2 States:** CO, FL, TX, VA, NH

As part of our launch, we want to identify and prioritize the top cities and zip codes to target barbershops and salons. We are looking to find locales where there is either **historically low voter turnout** or **underserved / disenfranchised populations**.

> **Note (Portfolio Version):** The original notebook connected to a live BigQuery database (`sutv.*` tables) and Google Cloud Storage. This version uses a synthetic dataset that mirrors the real schema, so the full analysis pipeline can be run end-to-end without credentials.


## 1. Setup — Import Libraries

In [ ]:
# Import Libraries and Set Options
import pandas as pd
import numpy as np
import json
import warnings
import os

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Libraries loaded successfully.")


## 2. Load Data

**Original pipeline:** Loaded `zipcodes.csv` (uploaded to Hex project filesystem) in chunks of 100,000 rows. Each row represented one US zip code enriched with Census demographic data — population by race, median household income, education attainment, and age distribution.

**Portfolio version:** A synthetic dataset is generated below using real zip codes and city names from the 13 target states, with plausible demographic values drawn from Census distributions.


In [ ]:
# ── Synthetic Data Generation ──────────────────────────────────────────────
# Real zip codes / cities from target swing states with synthetic demographic values.
# Schema mirrors the original zipcodes.csv enriched with Census API fields.

np.random.seed(42)

zip_data = [
    # (zipcode, city, state, population, pct_white, median_hh_income, pct_bachelors, median_age)
    # Georgia — Atlanta metro
    ("30301", "Atlanta",        "GA", 42000, 0.28, 38000, 0.31, 31),
    ("30303", "Atlanta",        "GA", 18000, 0.22, 34000, 0.28, 29),
    ("30310", "Atlanta",        "GA", 25000, 0.89, 72000, 0.55, 36),
    ("30318", "Atlanta",        "GA", 31000, 0.41, 51000, 0.40, 33),
    ("30324", "Atlanta",        "GA", 29000, 0.62, 63000, 0.50, 35),
    ("30060", "Marietta",       "GA", 38000, 0.45, 49000, 0.37, 34),
    ("30067", "Marietta",       "GA", 27000, 0.71, 68000, 0.53, 38),
    # Pennsylvania — Philadelphia metro
    ("19103", "Philadelphia",   "PA", 33000, 0.58, 61000, 0.52, 32),
    ("19121", "Philadelphia",   "PA", 28000, 0.19, 29000, 0.21, 28),
    ("19132", "Philadelphia",   "PA", 22000, 0.08, 25000, 0.15, 30),
    ("19140", "Philadelphia",   "PA", 36000, 0.12, 27000, 0.13, 29),
    ("19143", "Philadelphia",   "PA", 41000, 0.31, 33000, 0.24, 31),
    ("19401", "Norristown",     "PA", 19000, 0.44, 47000, 0.34, 33),
    # Michigan — Detroit metro
    ("48201", "Detroit",        "MI", 15000, 0.11, 22000, 0.18, 27),
    ("48205", "Detroit",        "MI", 24000, 0.05, 20000, 0.12, 28),
    ("48213", "Detroit",        "MI", 27000, 0.09, 23000, 0.14, 30),
    ("48227", "Detroit",        "MI", 31000, 0.07, 21000, 0.11, 29),
    ("48235", "Detroit",        "MI", 26000, 0.14, 28000, 0.19, 31),
    ("48033", "Southfield",     "MI", 34000, 0.22, 45000, 0.38, 36),
    # North Carolina — Charlotte metro
    ("28202", "Charlotte",      "NC", 20000, 0.49, 55000, 0.48, 30),
    ("28206", "Charlotte",      "NC", 29000, 0.31, 37000, 0.28, 31),
    ("28208", "Charlotte",      "NC", 33000, 0.27, 34000, 0.24, 30),
    ("28212", "Charlotte",      "NC", 37000, 0.38, 39000, 0.27, 32),
    ("28217", "Charlotte",      "NC", 28000, 0.43, 41000, 0.31, 33),
    # Ohio — Columbus metro
    ("43201", "Columbus",       "OH", 22000, 0.55, 42000, 0.43, 28),
    ("43205", "Columbus",       "OH", 19000, 0.24, 30000, 0.22, 27),
    ("43211", "Columbus",       "OH", 27000, 0.18, 27000, 0.16, 29),
    ("43219", "Columbus",       "OH", 31000, 0.33, 34000, 0.24, 31),
    # Arizona — Phoenix metro
    ("85003", "Phoenix",        "AZ", 17000, 0.41, 32000, 0.26, 29),
    ("85006", "Phoenix",        "AZ", 28000, 0.29, 29000, 0.18, 28),
    ("85008", "Phoenix",        "AZ", 34000, 0.35, 34000, 0.22, 31),
    ("85040", "Phoenix",        "AZ", 29000, 0.52, 47000, 0.31, 33),
    # Nevada — Las Vegas metro
    ("89101", "Las Vegas",      "NV", 26000, 0.30, 28000, 0.17, 30),
    ("89104", "Las Vegas",      "NV", 31000, 0.32, 31000, 0.19, 31),
    ("89106", "Las Vegas",      "NV", 24000, 0.27, 26000, 0.15, 29),
    ("89121", "Las Vegas",      "NV", 38000, 0.44, 38000, 0.25, 33),
    # Wisconsin — Milwaukee metro
    ("53202", "Milwaukee",      "WI", 18000, 0.72, 57000, 0.51, 32),
    ("53206", "Milwaukee",      "WI", 21000, 0.05, 22000, 0.11, 28),
    ("53209", "Milwaukee",      "WI", 27000, 0.12, 27000, 0.14, 30),
    ("53215", "Milwaukee",      "WI", 31000, 0.20, 31000, 0.18, 31),
    # Florida — Miami/Tampa
    ("33101", "Miami",          "FL", 29000, 0.18, 30000, 0.21, 30),
    ("33125", "Miami",          "FL", 37000, 0.12, 27000, 0.15, 29),
    ("33602", "Tampa",          "FL", 22000, 0.50, 48000, 0.40, 31),
    ("33610", "Tampa",          "FL", 34000, 0.27, 31000, 0.19, 30),
    # Texas — Houston/Dallas
    ("77002", "Houston",        "TX", 20000, 0.32, 38000, 0.36, 29),
    ("77051", "Houston",        "TX", 31000, 0.08, 24000, 0.13, 28),
    ("75201", "Dallas",         "TX", 25000, 0.46, 55000, 0.48, 30),
    ("75216", "Dallas",         "TX", 38000, 0.11, 26000, 0.14, 29),
    # Virginia — Richmond/NoVA
    ("23220", "Richmond",       "VA", 22000, 0.38, 42000, 0.38, 28),
    ("23224", "Richmond",       "VA", 27000, 0.23, 30000, 0.20, 30),
    ("22201", "Arlington",      "VA", 30000, 0.61, 88000, 0.72, 33),
    # Colorado — Denver metro
    ("80204", "Denver",         "CO", 25000, 0.36, 40000, 0.34, 30),
    ("80216", "Denver",         "CO", 29000, 0.27, 34000, 0.22, 31),
    ("80229", "Thornton",       "CO", 33000, 0.44, 52000, 0.30, 33),
    # New Hampshire — Manchester
    ("03101", "Manchester",     "NH", 18000, 0.68, 48000, 0.32, 33),
    ("03103", "Manchester",     "NH", 22000, 0.60, 44000, 0.28, 32),
]

cols = ["zipcode", "city", "state", "population_by_year_2018",
        "population_by_race_White_pct", "median_household_income",
        "education_bachelors_pct", "median_age"]

zipcodes = pd.DataFrame(zip_data, columns=cols)

# Derive raw race counts from percentages (mirrors original Census API format)
zipcodes["population_by_race_White"] = (
    zipcodes["population_by_year_2018"] * zipcodes["population_by_race_White_pct"]
).astype(int).astype(str)

zipcodes["population_by_race_total"] = zipcodes["population_by_year_2018"].astype(str)

# Add county column (cleaned — no "County" suffix, mirrors cleaning step)
county_map = {
    "GA": "Fulton", "PA": "Philadelphia", "MI": "Wayne",
    "NC": "Mecklenburg", "OH": "Franklin", "AZ": "Maricopa",
    "NV": "Clark", "WI": "Milwaukee", "FL": "Miami-Dade",
    "TX": "Harris", "VA": "Richmond City", "CO": "Denver", "NH": "Hillsborough"
}
zipcodes["county"] = zipcodes["state"].map(county_map)

# Simulate common_city_list column (originally a stringified list from Census)
zipcodes["common_city_list"] = zipcodes["city"].apply(lambda c: f"['{c}', '{c} Heights']")

print(f"Loaded {len(zipcodes)} zip codes across {zipcodes['state'].nunique()} states.")
zipcodes.head(8)


## 3. Data Cleaning

The raw zip code data required several cleaning steps before analysis:
- **`common_city_list`**: Stored as a stringified list — extract only the primary city name
- **`county`**: Strip the word "County" from county names for consistency


In [ ]:
# Clean common_city_list — extract primary city only (mirrors original chunk-level cleaning)
split_columns = zipcodes['common_city_list'].str.split(',', expand=True)
zipcodes['common_city_list'] = (
    split_columns[0]
    .str.replace('[', '', regex=False)
    .str.replace(']', '', regex=False)
    .str.replace("'", '', regex=False)
    .str.strip()
)

# Clean county column — remove "County" suffix
zipcodes['county'] = zipcodes['county'].str.replace("County", '', regex=False).str.strip()

print("Cleaned columns:")
print(zipcodes[['zipcode', 'common_city_list', 'county', 'state']].head(8).to_string(index=False))


### Column Types

In [ ]:
# Inspect data types (sorted alphabetically — mirrors original exploration step)
zipcodes.dtypes.sort_index()


## 4. Feature Engineering — Demographic Scoring

### Hypothesis

> *Young people, as well as people with less education and lower incomes, will be less likely to cast a ballot in any given election.*  
> Source: [Elections Canada — Correlates of Voter Turnout](https://www.elections.ca/content.aspx?section=res&dir=rec/part/tud&document=correlates&lang=e)

Based on this hypothesis, we flag zip codes as **high-priority** if their demographics suggest lower civic engagement or underrepresentation.


In [ ]:
warnings.simplefilter(action='ignore', category=Warning)

def calc_nonwhite_percentage(row) -> float:
    """
    Calculate the percentage of non-white residents in a zip code.
    Returns None if race data is missing.
    """
    if row['population_by_race_White'] == '' or row['population_by_race_total'] == '':
        return None
    return 1 - int(row['population_by_race_White']) / int(row['population_by_race_total'])

# Apply non-white percentage calculation
exp_df = zipcodes.copy()
exp_df['population_by_race_nonwhite_percentage'] = exp_df.apply(
    lambda row: calc_nonwhite_percentage(row), axis=1
)

# Flag high-priority zips: >33% non-white population
# (Threshold chosen to identify communities of color that are historically underserved)
exp_df['high_priority_zip'] = exp_df['population_by_race_nonwhite_percentage'] > 0.33

print(f"High-priority zips: {exp_df['high_priority_zip'].sum()} / {len(exp_df)}")
exp_df[['zipcode', 'city', 'state', 'population_by_race_nonwhite_percentage', 'high_priority_zip']].head(10)


## 5. Export Results

**Original pipeline:** Results were uploaded directly to a Google Cloud Storage bucket (`shape-up-the-vote-data/high_priority_zips.csv`) using a service account.

**Portfolio version:** Results are saved locally to `high_priority_zips.csv`.


In [ ]:
# Save results locally (replaces GCS upload)
output_path = "high_priority_zips.csv"
exp_df.to_csv(output_path, index=False)
print(f"Saved {len(exp_df)} rows to '{output_path}'")


## 6. State-Level Summary

How many zip codes in each target state are flagged as high-priority vs. standard?


In [ ]:
# Pivot: count high-priority vs standard zips per state
pivoted_df = exp_df.pivot_table(
    index="state", columns="high_priority_zip", aggfunc="size", fill_value=0
).reset_index()

pivoted_df.columns = ["state", "standard_priority", "high_priority"]
pivoted_df["total_zips"] = pivoted_df["standard_priority"] + pivoted_df["high_priority"]
pivoted_df["pct_high_priority"] = (pivoted_df["high_priority"] / pivoted_df["total_zips"] * 100).round(1)

target_states = ['GA', 'PA', 'NC', 'MI', 'OH', 'NV', 'TX', 'FL', 'NH', 'AZ', 'CO', 'VA', 'WI', 'WI']
result = pivoted_df[pivoted_df['state'].isin(target_states)].sort_values("pct_high_priority", ascending=False)
result


### Visualization — High-Priority Zip Share by State

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

fig, ax = plt.subplots(figsize=(10, 5))

colors = ["#E45756" if p >= 50 else "#4C78A8" for p in result["pct_high_priority"]]
bars = ax.barh(result["state"], result["pct_high_priority"], color=colors)

ax.set_xlabel("% of Zip Codes Flagged High-Priority", fontsize=11)
ax.set_title("High-Priority Zip Code Share by State\n(>33% non-white population)", fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.axvline(50, color='gray', linestyle='--', linewidth=0.8, label='50% threshold')
ax.legend(fontsize=9)

for bar, val in zip(bars, result["pct_high_priority"]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig("high_priority_by_state.png", dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved.")
